# 02 - Reporte de calidad de datos | RADAR Cibest

**Fase ASUM-DM:** 2 - Entendimiento de datos  
**Objetivo:** Perfilado estadistico de cada variable y deteccion de paises con cobertura insuficiente

In [60]:
import sys
from pathlib import Path
import re
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from src.utils import load_all_configs, resolve_data_path, get_variable_catalog
from src.data_preparation.cleaning import pivot_latest_value_and_year, apply_freshness_filter


catalog = get_variable_catalog(configs["variables"])

configs = load_all_configs()
raw_dir = resolve_data_path(configs['settings']['data']['raw_path'])



pattern = re.compile(r"^master_raw_\d{8}\.parquet$")

candidates = sorted(
    [
        p for p in raw_dir.glob("master_raw_*.parquet")
        if pattern.match(p.name)
    ],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise FileNotFoundError("Ejecute primero el pipeline de extraccion")

master = pd.read_parquet(candidates[0])

print(f"Archivo cargado: {candidates[0].name}")
print(f"Master cargado: {master.shape}")


Archivo cargado: master_raw_20260612.parquet
Master cargado: (1281, 5)


## 1. Perfilado por variable

In [61]:
n_countries = master["country_iso3"].nunique()

stats = (
    master.dropna(subset=["value"])
    .groupby("variable")["value"]
    .agg(
        n_obs="count",
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

coverage_var = (
    master.dropna(subset=["value"])
    .groupby("variable")["country_iso3"]
    .nunique()
    .reset_index(name="n_countries_with_data")
)

summary = stats.merge(coverage_var, on="variable", how="left")

summary["n_countries_expected"] = n_countries
summary["n_missing_countries"] = (
    summary["n_countries_expected"] - summary["n_countries_with_data"]
)
summary["pct_coverage"] = (
    summary["n_countries_with_data"] / summary["n_countries_expected"]
)
summary["pct_missing"] = 1 - summary["pct_coverage"]

summary = summary.sort_values("pct_missing", ascending=False)

summary_display = summary.copy()

summary_display["pct_coverage"] = (summary_display["pct_coverage"] * 100).round(1)
summary_display["pct_missing"] = (summary_display["pct_missing"] * 100).round(1)

summary_display[[
    "variable",
    "n_obs",
    "n_countries_with_data",
    "n_missing_countries",
    "pct_coverage",
    "pct_missing",
    "mean",
    "std",
    "min",
    "max"
]].style.format({
    "n_obs": "{:,.0f}",
    "n_countries_with_data": "{:,.0f}",
    "n_missing_countries": "{:,.0f}",
    "pct_coverage": "{:.1f}%",
    "pct_missing": "{:.1f}%",
    "mean": "{:,.2f}",
    "std": "{:,.2f}",
    "min": "{:,.2f}",
    "max": "{:,.2f}",
}).background_gradient(
    subset=["pct_coverage"],
    cmap="RdYlGn"
).background_gradient(
    subset=["pct_missing"],
    cmap="YlOrRd"
)

,variable,n_obs,n_countries_with_data,n_missing_countries,pct_coverage,pct_missing,mean,std,min,max
22,hofstede_ivr,16,16,14,53.3%,46.7%,67.06,18.05,44.00,100.00
39,stock_market_cap_gdp,18,18,12,60.0%,40.0%,44.21,56.26,1.38,216.29
23,hofstede_lto,18,18,12,60.0%,40.0%,23.33,14.99,0.00,54.00
35,public_debt_gdp,19,19,11,63.3%,36.7%,64.88,34.99,13.15,125.13
21,hofstede_idv,23,23,7,76.7%,23.3%,35.35,17.78,11.00,72.00
2,bank_capital_rwa,23,23,7,76.7%,23.3%,17.06,2.89,12.70,25.60
4,bank_npl_ratio,23,23,7,76.7%,23.3%,2.62,1.34,0.46,5.84
5,colombian_diaspora_stock,23,23,7,76.7%,23.3%,"11,013.04","17,275.88",23.00,"61,020.00"
26,hofstede_uai,23,23,7,76.7%,23.3%,74.30,21.50,13.00,98.00
25,hofstede_pdi,23,23,7,76.7%,23.3%,65.65,17.29,35.00,95.00


## 2. Matriz de cobertura por pais y variable

In [62]:
coverage_matrix = master.dropna(subset=['value']).groupby(['country_iso3', 'variable']).size().unstack(fill_value=0)
coverage_matrix = (coverage_matrix > 0).astype(int)
print(f'Cobertura media: {coverage_matrix.mean().mean()*100:.1f}%')
coverage_matrix#.head()

Cobertura media: 90.4%


variable,account_ownership,atms_per_100k_adults,bank_capital_rwa,bank_concentration_5,bank_npl_ratio,colombian_diaspora_stock,commercial_bank_branches_per_100k_adults,common_language_spanish,control_of_corruption,country_risk_premium,...,public_debt_gdp,regulatory_quality,rule_of_law,secure_internet_servers_per_million,stock_market_cap_gdp,trade_openness,unemployment_rate,urban_population_pct,voice_accountability,weighted_mean_applied_tariff_all_products
country_iso3,,,,,,,,,,,,,,,,,,,,,
ARG,1,1,1,1,1,1,1,1,1,1,...,0,1,1,1,1,1,1,1,1,1
BHS,0,1,0,1,0,1,1,1,1,1,...,1,1,1,1,0,1,1,1,1,1
BLZ,1,1,0,1,1,0,1,1,1,1,...,1,1,1,1,0,1,1,1,1,1
BOL,1,1,1,1,1,0,1,1,1,1,...,1,1,1,1,0,1,1,1,1,1
BRA,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
BRB,0,1,0,1,1,1,1,1,1,1,...,1,1,1,1,1,0,1,1,1,1
CAN,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
CHL,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
COL,1,1,1,1,1,0,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [63]:
import plotly.express as px
fig = px.imshow(coverage_matrix.T, color_continuous_scale=['#FFD0D0', '#0D1B2A'],
                aspect='auto', title='Matriz de cobertura: pais x variable')
fig.update_layout(height=900)
fig.show()

In [64]:
#variables y países están siendo castigados por datos antiguos.

wide_values, wide_years = pivot_latest_value_and_year(master)

wide_fresh, stale_report = apply_freshness_filter(
    wide_values=wide_values,
    wide_years=wide_years,
    variable_catalog=catalog,
    settings=configs["settings"],
)

stale_cells = (
    wide_values.notna()
    & wide_fresh.isna()
).sum().sum()

print("Celdas marcadas como stale:", stale_cells)

stale_by_variable = (
    wide_values.notna()
    & wide_fresh.isna()
).sum(axis=0).sort_values(ascending=False)

stale_by_country = (
    wide_values.notna()
    & wide_fresh.isna()
).sum(axis=1).sort_values(ascending=False)

display(stale_by_variable)
display(stale_by_country)
display(stale_report.sort_values("n_stale_variables", ascending=False))

2026-06-12 19:59:12.533 | INFO     | src.data_preparation.cleaning:pivot_latest_value_and_year:133 - Pivoteo ancho: 30 paises x 45 variables (estrategia=latest_available)
2026-06-12 19:59:12.552 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:200 - Control de antigüedad aplicado: reference_year=2026, max_age=5, celdas stale=100
2026-06-12 19:59:12.559 | INFO     | src.data_preparation.cleaning:apply_freshness_filter:225 - Variables con más datos stale: {'bank_capital_rwa': 23, 'colombian_diaspora_stock': 23, 'public_debt_gdp': 9, 'stock_market_cap_gdp': 6, 'financial_system_deposits_gdp': 6, 'bank_concentration_5': 5, 'interest_rate_spread': 4, 'ict_goods_exports_pct_total_goods_exports': 4, 'atms_per_100k_adults': 3, 'personal_remittances_gdp': 2, 'current_account_gdp': 2, 'digital_payment_adults_pct': 2, 'domestic_credit_private_gdp': 2, 'commercial_bank_branches_per_100k_adults': 2, 'trade_openness': 2}
2026-06-12 19:59:12.563 | INFO     | src.data_preparation.clea

Celdas marcadas como stale: 100


variable
bank_capital_rwa                             23
colombian_diaspora_stock                     23
public_debt_gdp                               9
stock_market_cap_gdp                          6
financial_system_deposits_gdp                 6
bank_concentration_5                          5
interest_rate_spread                          4
ict_goods_exports_pct_total_goods_exports     4
atms_per_100k_adults                          3
personal_remittances_gdp                      2
current_account_gdp                           2
digital_payment_adults_pct                    2
domestic_credit_private_gdp                   2
commercial_bank_branches_per_100k_adults      2
trade_openness                                2
inflation_rate                                1
account_ownership                             1
weighted_mean_applied_tariff_all_products     1
gdp_per_capita_ppp                            1
gdp_nominal                                   1
geographic_distance_km         

country_iso3
VEN    12
BRB     7
HTI     6
TTO     5
CAN     5
CHL     4
PAN     4
CUB     4
GTM     3
USA     3
URY     3
PRY     3
JAM     3
BHS     3
ARG     3
ECU     3
DOM     3
CRI     3
ESP     2
HND     2
MEX     2
NIC     2
PER     2
SLV     2
SUR     2
BRA     2
BOL     2
BLZ     2
GUY     2
COL     1
dtype: int64

,n_stale_variables,pct_stale_variables
country_iso3,,
VEN,12,0.266667
BRB,7,0.155556
HTI,6,0.133333
TTO,5,0.111111
CAN,5,0.111111
CHL,4,0.088889
PAN,4,0.088889
CUB,4,0.088889
GTM,3,0.066667
